In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
BASE_DIR = Path(r"D:\Downloads\Capstone\Dataset\HDFS_v1\preprocessed")

TRACE_FILE = BASE_DIR / "Event_traces.csv"
LABEL_FILE = BASE_DIR / "anomaly_label.csv"

OUTPUT_DIR = Path("./tfidf_rf_hdfs_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Trace file exists:", TRACE_FILE.exists(), TRACE_FILE)
print("Label file exists:", LABEL_FILE.exists(), LABEL_FILE)

Trace file exists: False D:\Downloads\Capstone\Dataset\HDFS_v1\preprocessed\Event_traces.csv
Label file exists: False D:\Downloads\Capstone\Dataset\HDFS_v1\preprocessed\anomaly_label.csv


In [3]:
trace_sample = pd.read_csv(TRACE_FILE, nrows=5)
label_sample = pd.read_csv(LABEL_FILE, nrows=5)

print("Event_traces columns:")
print(trace_sample.columns.tolist())
display(trace_sample)

print("\nanomaly_label columns:")
print(label_sample.columns.tolist())
display(label_sample)

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Downloads\\Capstone\\Dataset\\HDFS_v1\\preprocessed\\Event_traces.csv'

In [4]:
traces_df = pd.read_csv(TRACE_FILE)
labels_df = pd.read_csv(LABEL_FILE)

print("Traces shape:", traces_df.shape)
print("Labels shape:", labels_df.shape)

Traces shape: (575061, 6)
Labels shape: (575061, 2)


In [5]:
traces_df.columns = [c.strip() for c in traces_df.columns]
labels_df.columns = [c.strip() for c in labels_df.columns]

print("Trace columns:", traces_df.columns.tolist())
print("Label columns:", labels_df.columns.tolist())

Trace columns: ['BlockId', 'Label', 'Type', 'Features', 'TimeInterval', 'Latency']
Label columns: ['BlockId', 'Label']


In [6]:
possible_block_cols = ["BlockId", "BlockID", "Block_Id", "blk_id"]
possible_trace_cols = ["EventSequence", "EventSeq", "Sequence", "Events", "EventIdSequence", "EventIds"]
possible_label_cols = ["Label", "label", "Target", "target", "Class"]

block_col_trace = next((c for c in possible_block_cols if c in traces_df.columns), None)
trace_col = next((c for c in possible_trace_cols if c in traces_df.columns), None)

block_col_label = next((c for c in possible_block_cols if c in labels_df.columns), None)
label_col = next((c for c in possible_label_cols if c in labels_df.columns), None)

print("Detected trace BlockId column:", block_col_trace)
print("Detected event sequence column:", trace_col)
print("Detected label BlockId column:", block_col_label)
print("Detected label column:", label_col)

Detected trace BlockId column: BlockId
Detected event sequence column: None
Detected label BlockId column: BlockId
Detected label column: Label


In [7]:
print("Event_traces columns:")
print(traces_df.columns.tolist())

display(traces_df.head(10))

for col in traces_df.columns:
    print(f"\n--- Column: {col} ---")
    print(traces_df[col].astype(str).head(5).tolist())

Event_traces columns:
['BlockId', 'Label', 'Type', 'Features', 'TimeInterval', 'Latency']


,BlockId,Label,Type,Features,TimeInterval,Latency
0,blk_-1608999687919862906,Success,NaN,"[E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26...","[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3802
1,blk_7503483334202473044,Success,NaN,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3802
2,blk_-3544583377289625738,Fail,21.0,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E3,E26,E26,...","[0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ...",3797
3,blk_-9073992586687739851,Success,NaN,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",50448
4,blk_7854771516489510256,Success,NaN,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[0.0, 0.0, 1.0, 48.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",50583
5,blk_1717858812220360316,Success,NaN,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[0.0, 0.0, 11.0, 43.0, 0.0, 0.0, 0.0, 0.0, 0.0...",50458
6,blk_-2519617320378473615,Success,NaN,"[E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26...","[0.0, 1.0, 9.0, 43.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",50523
7,blk_7063315473424667801,Success,NaN,"[E5,E5,E5,E22,E11,E9,E11,E9,E26,E26,E11,E9,E26...","[1.0, 0.0, 0.0, 58.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",50818
8,blk_8586544123689943463,Success,NaN,"[E5,E5,E5,E22,E11,E9,E11,E11,E9,E9,E26,E26,E26...","[1.0, 0.0, 0.0, 36.0, 0.0, 1.0, 0.0, 0.0, 0.0,...",50795
9,blk_2765344736980045501,Success,NaN,"[E5,E5,E22,E5,E11,E9,E11,E9,E26,E11,E9,E26,E26...","[0.0, 0.0, 1.0, 28.0, 0.0, 0.0, 0.0, 0.0, 1.0,...",50528



--- Column: BlockId ---
['blk_-1608999687919862906', 'blk_7503483334202473044', 'blk_-3544583377289625738', 'blk_-9073992586687739851', 'blk_7854771516489510256']

--- Column: Label ---
['Success', 'Success', 'Fail', 'Success', 'Success']

--- Column: Type ---
['nan', 'nan', '21.0', 'nan', 'nan']

--- Column: Features ---
['[E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26,E6,E5,E16,E6,E5,E18,E25,E26,E26,E3,E25,E6,E6,E5,E5,E16,E18,E26,E26,E5,E6,E5,E16,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E18,E25,E6,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E26,E26,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E25,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E18,E6,E5,E3,E3,E3,E3,E3,E16,E3,E3,E3,E3,E26,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E

In [8]:
def normalize_event_text(x):
    x = str(x).strip()
    x = x.replace("[", " ").replace("]", " ")
    x = x.replace(",", " ")
    x = x.replace("'", " ").replace('"', " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

traces_df["text"] = traces_df["Features"].map(normalize_event_text)

print(traces_df[["BlockId", "Features", "text"]].head(5))

                    BlockId  \
0  blk_-1608999687919862906   
1   blk_7503483334202473044   
2  blk_-3544583377289625738   
3  blk_-9073992586687739851   
4   blk_7854771516489510256   

                                            Features  \
0  [E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26...   
1  [E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...   
2  [E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E3,E26,E26,...   
3  [E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...   
4  [E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...   

                                                text  
0  E5 E22 E5 E5 E11 E11 E9 E9 E11 E9 E26 E26 E26 ...  
1  E5 E5 E22 E5 E11 E9 E11 E9 E11 E9 E26 E26 E26 ...  
2  E5 E22 E5 E5 E11 E9 E11 E9 E11 E9 E3 E26 E26 E...  
3  E5 E22 E5 E5 E11 E9 E11 E9 E11 E9 E26 E26 E26 ...  
4  E5 E5 E22 E5 E11 E9 E11 E9 E11 E9 E26 E26 E26 ...  


In [10]:
labels_df["Label"] = labels_df["Label"].astype(str).str.strip().str.lower()

label_map = {
    "normal": 0,
    "anomaly": 1
}

labels_df["target"] = labels_df["Label"].map(label_map)

print(labels_df[["BlockId", "Label", "target"]].head())
print(labels_df["target"].value_counts(dropna=False))

                    BlockId    Label  target
0  blk_-1608999687919862906   normal       0
1   blk_7503483334202473044   normal       0
2  blk_-3544583377289625738  anomaly       1
3  blk_-9073992586687739851   normal       0
4   blk_7854771516489510256   normal       0
target
0    558223
1     16838
Name: count, dtype: int64


In [11]:
labels_small = labels_df[["BlockId", "target"]].copy()

dataset_df = traces_df[["BlockId", "text"]].merge(
    labels_small,
    on="BlockId",
    how="inner"
)

dataset_df = dataset_df.drop_duplicates(subset=["BlockId"]).reset_index(drop=True)

print("Merged dataset shape:", dataset_df.shape)
print(dataset_df["target"].value_counts())
print(dataset_df.head())

Merged dataset shape: (575061, 3)
target
0    558223
1     16838
Name: count, dtype: int64
                    BlockId  \
0  blk_-1608999687919862906   
1   blk_7503483334202473044   
2  blk_-3544583377289625738   
3  blk_-9073992586687739851   
4   blk_7854771516489510256   

                                                text  target  
0  E5 E22 E5 E5 E11 E11 E9 E9 E11 E9 E26 E26 E26 ...       0  
1  E5 E5 E22 E5 E11 E9 E11 E9 E11 E9 E26 E26 E26 ...       0  
2  E5 E22 E5 E5 E11 E9 E11 E9 E11 E9 E3 E26 E26 E...       1  
3  E5 E22 E5 E5 E11 E9 E11 E9 E11 E9 E26 E26 E26 ...       0  
4  E5 E5 E22 E5 E11 E9 E11 E9 E11 E9 E26 E26 E26 ...       0  


In [12]:
X = dataset_df["text"]
y = dataset_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))
print("\nTrain label distribution:")
print(y_train.value_counts())
print("\nTest label distribution:")
print(y_test.value_counts())

Train size: 460048
Test size : 115013

Train label distribution:
target
0    446578
1     13470
Name: count, dtype: int64

Test label distribution:
target
0    111645
1      3368
Name: count, dtype: int64


In [13]:
tfidf = TfidfVectorizer(
    max_features=10000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 1),
    dtype=np.float32
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape :", X_test_tfidf.shape)
print("Vocabulary size   :", len(tfidf.get_feature_names_out()))

Train TF-IDF shape: (460048, 24)
Test TF-IDF shape : (115013, 24)
Vocabulary size   : 24


In [5]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X_train_tfidf, y_train)

print("Random Forest training completed.")

NameError: name 'X_train_tfidf' is not defined

In [15]:
y_pred = rf.predict(X_test_tfidf)

print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1-score :", round(f1_score(y_test, y_pred, zero_division=0), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

Accuracy : 0.8318
Precision: 0.1482
Recall   : 0.9994
F1-score : 0.2582

Confusion Matrix:
[[92302 19343]
 [    2  3366]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.8267    0.9051    111645
           1     0.1482    0.9994    0.2582      3368

    accuracy                         0.8318    115013
   macro avg     0.5741    0.9131    0.5817    115013
weighted avg     0.9750    0.8318    0.8862    115013



In [16]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_prob = rf.predict_proba(X_test_tfidf)[:, 1]

thresholds = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    f = f1_score(y_test, y_pred_t, zero_division=0)
    print(f"Threshold={t:.2f} | Precision={p:.4f} | Recall={r:.4f} | F1={f:.4f}")

Threshold=0.50 | Precision=0.1482 | Recall=0.9994 | F1=0.2582
Threshold=0.60 | Precision=0.9982 | Recall=0.8257 | F1=0.9038
Threshold=0.70 | Precision=0.9982 | Recall=0.8257 | F1=0.9038
Threshold=0.80 | Precision=0.9982 | Recall=0.8251 | F1=0.9034
Threshold=0.90 | Precision=0.9982 | Recall=0.8236 | F1=0.9026
Threshold=0.95 | Precision=0.9989 | Recall=0.8201 | F1=0.9007
Threshold=0.99 | Precision=0.9992 | Recall=0.7883 | F1=0.8813


The initial Random Forest model using the default classification threshold of 0.50 produced very high recall but low precision, meaning that most anomalies were detected but many normal instances were incorrectly classified as anomalies. To improve the balance between false positives and false negatives, threshold tuning was applied using the predicted probabilities. A threshold of 0.60 gave the best overall result, achieving a precision of 0.9982, recall of 0.8257, and F1-score of 0.9038. This indicates that the model performs well when the decision threshold is adjusted appropriately for the imbalanced nature of the dataset.

The TF-IDF + Random Forest approach was used as a lightweight traditional baseline to provide a reference point for anomaly detection 
performance. However, because it represents logs as frequency-based features, it cannot fully capture event order and context. LogBERT was 
selected as the main model because it is better aligned with the sequential nature of log data. While the evaluation settings were not fully 
identical across both models, the overall findings support the use of LogBERT as the more appropriate approach for this project.